# 🧩 Feature Engineering — ChurnGuard AI

Prototype, inspect and **validate** the business-oriented engineered features. The reusable implementation lives in `src/feature_engineering.py` and is the single source of truth used at train and inference time.

**Engineered features**
| Feature | Idea |
|---|---|
| `tenure_group` | lifecycle stage from Account length |
| `service_count` | number of subscribed plans |
| `total_charge` | total spend across all periods |
| `avg_charge_per_minute` | pricing-efficiency proxy |
| `customer_value_score` | approximate lifetime value |
| `service_call_risk` | support-contact intensity |

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (9, 5)

from src.data_preprocessing import load_clean_split
from src.feature_engineering import add_features, ENGINEERED_FEATURES

train, test = load_clean_split()
raw = pd.concat([train, test], ignore_index=True)
df = add_features(raw)
print('Engineered:', ENGINEERED_FEATURES)
df[ENGINEERED_FEATURES + ['Churn']].head()

## 1. tenure_group — lifecycle stage

Account length bucketed into business-friendly stages (New → Veteran).

In [ ]:
order = ['New', 'Growing', 'Established', 'Loyal', 'Veteran']
grp = df.groupby('tenure_group')['Churn'].agg(['mean', 'count']).reindex(order)
display(grp)
(grp['mean'] * 100).plot(kind='bar', color='#2563eb',
                          title='Churn rate by tenure_group (%)')
plt.ylabel('Churn rate (%)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 2. service_count — plan engagement

Count of subscribed plans (International + Voice mail). More engaged customers tend to be stickier.

In [ ]:
display(df.groupby('service_count')['Churn'].agg(['mean', 'count']))
(df.groupby('service_count')['Churn'].mean() * 100).plot(
    kind='bar', color='#16a34a', title='Churn rate by service_count (%)')
plt.ylabel('Churn rate (%)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3. service_call_risk — strongest driver

`Customer service calls / 4`. Values ≥ 1 mark the high-risk zone identified in the EDA.

In [ ]:
df['high_risk_zone'] = df['service_call_risk'] >= 1.0
print(df.groupby('high_risk_zone')['Churn'].mean())

plt.scatter(df['service_call_risk'], df['Churn'], alpha=0.05, color='#dc2626')
binned = df.groupby('service_call_risk')['Churn'].mean()
plt.plot(binned.index, binned.values, color='black', marker='o', label='mean churn')
plt.title('service_call_risk vs churn'); plt.xlabel('service_call_risk')
plt.ylabel('Churn'); plt.legend(); plt.tight_layout(); plt.show()

## 4. Spending features — total_charge, avg_charge_per_minute, customer_value_score

In [ ]:
spend = ['total_charge', 'avg_charge_per_minute', 'customer_value_score']
display(df.groupby('Churn')[spend].mean().T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, spend):
    for label, color in [(0, '#16a34a'), (1, '#dc2626')]:
        ax.hist(df[df['Churn'] == label][col], bins=30, alpha=0.55, color=color,
                label='Churned' if label else 'Retained')
    ax.set_title(col); ax.legend()
plt.tight_layout(); plt.show()

## 5. Do the engineered features add signal?

Correlation of each engineered numeric feature with the target.

In [ ]:
numeric_engineered = ['service_count', 'total_charge', 'avg_charge_per_minute',
                      'customer_value_score', 'service_call_risk']
df[numeric_engineered + ['Churn']].corr()['Churn'].drop('Churn').sort_values(
    ascending=False)

In [ ]:
# Quick sanity check: a simple model on engineered features alone should beat
# the base rate, confirming the features carry signal.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_tr = add_features(train)[numeric_engineered]
X_te = add_features(test)[numeric_engineered]
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=42).fit(X_tr, train['Churn'])
auc = roc_auc_score(test['Churn'], rf.predict_proba(X_te)[:, 1])
print(f'ROC-AUC using ONLY engineered features: {auc:.3f}')

## 6. Summary

- `service_call_risk` and `total_charge` carry the most standalone signal.
- `tenure_group` and `service_count` add interpretable, business-facing segmentation.
- Engineered features alone already give a strong ROC-AUC, validating the design.

The full feature set (raw + engineered) is encoded/scaled inside the model pipeline in `src/train_model.py`.